# Inteligencia Artificial
### Parte práctica de la primera prueba de evaluación alternativa. Curso 2022-23
### Grupo 1

In [ ]:
# Apellidos:
# Nombre:

Un sistema de maximización basado en enjambre de partículas consta de $n$ partículas que ocupan una posición en $R^m$ y que llevan asociado un vector de velocidad, junto con una función fitness que asocia un valor a cada
posición. El objetivo es encontrar la posición donde la función fitness toma el valor máximo. Los siguientes ejercicios van orientados a la implementación y evaluación de un sistema de este tipo.

In [3]:
import random
import math

El comportamiento de una partícula en este tipo de sistemas viene dado por:

1. La posición y velocidad inicial de las partículas se establece aleatoriamente.
2. Todas las partículas actualizan su posición en paralelo.
3. Para cada partícula, su posición en el instante $t + 1$ responde a la siguiente fórmula: $ x_i^{t+1} = x_i^t + v_i^{t+1}$

4. Para cada partícula, su velocidad en el instante $t + 1$ responde a la siguiente fórmula:

$$ v_i^{t+1} = W v_i^t + c_1 r_1 (m_i^t  - x_i^t) + c_2 r_2 (b^t - x_i^t)$$

Donde $W \in R$ representa la inercia,  $c_1$ representa la influencia individual, $c_2$ representa la influencia social, $r_1$ y $r_2$ son dos valores aleatorios en el intervalo $[0, 1]$ y $m_i^t$ y $b^t$ representan la mejor posición de la partícula $i$ en los pasos $0, . . . , t$ y la mejor posición de cualquier partícula $i$ en los pasos $0, . . . , t$ respectivamente.

### Ejercicio 1 [1.5 ptos]
Implementa la clase `Partícula`, con el método constructor y un método llamado `actualiza` que permita actualizar la posición y velocidad de la partícula. En concreto:

* El método constructor debe recibir los parámetros del sistema asociados al comportamiento de la partícula (`W`, `c1`, `c2`) así como la función a optimizar `f`. Además, debe recibir el número de dimensiones `ndims` del espacio en el que se mueven las partículas (y donde está definida `f`), y un parámetro `a` que sirve para acotar la posición inicial aleatoria de la partícula (cuyas coordenadas iniciales serán números aleatorios en el intervalo $[-a,a]$, usar `random.unifom` para ello). El constructor también debe inicializar la velocidad inicial de la partícula a cero. Supondremos que tanto los vectores de posición como los de velocidad se representan mediante listas de longitud `ndim`. Por último, la clase debe poseer atributos para almacenar el mejor valor de la partícula hasta el momento `self.best`, y la posición donde ha tenido ese mejor valor `self.bestpos`.

* El método `actualiza` recibe las coordenadas del mejor global `bestglobal` y debe actualizar la posición y velocidad de la partícula así como (si fuera el caso), la mejor posición de la partícula y su valor asociado. Se espera que este método devuelva el valor de estos dos últimos atributos. 

**Nota**: En los siguientes ejercicios, el estudiante puede añadir y/o modificar el código que considere necesario, más allá de completar los huecos indicados para ello.

In [4]:
class Partícula():
    
    def __init__(self,W,c1,c2,f,a,ndims):
        self.a=a
        self.ndims=ndims
        self.pos = [random.uniform(-(self.a),self.a) for _ in range (self.ndims)]
        self.W,self.c1,self.c2,self.f = W,c1,c2,f
        self.vel = [0] * self.ndims # [0,0,0,...]
        self.best = -float("infinity")
        self.bestpos= self.pos
        
    def actualiza(self,bestglobal):
        self.pos = [(x+y) for x,y in zip (self.pos,self.vel)]
        r1=random.uniform(0,1)
        r2=random.uniform(0,1)
        self.vel = [self.W*vi + self.c1*r1*(bpi-xi)+self.c2*r2*(bi-xi) for (xi,vi,bpi,bi) in zip(self.pos,self.vel,self.bestpos,bestglobal)]
        xd=[x+y for x,y in zip (self.pos,self.vel)]
        if self.f(xd) > self.f(self.pos):
            self.best, self.bestpos= self.f(xd),xd
        return self.best,self.bestpos

### Ejercicio 2 [1.5 ptos]

Implementa la clase `PSO`, con el método constructor, un método llamado `actualiza_enjambre` que permita actualizar la posición y velocidad de todas las partículas en el sistema y un método llamado `ejecutar` que lleve a cabo la ejecución del algoritmo y devuelva las coordenadas de la mejor posición encontrada y del valor de la función de la valoración en esa posición.

* El método constructor debe recibir los siguientes parámetros: función de valoración,número de partículas en el enjambre, número de iteraciones, `W`,`c1`, `c2`, `a` y `ndims`. El constructor debe crear un enjambre con tantas partículas como indique el parámetro $nparticles$.

* El método `actualiza_enjambre` debe actualizar la posición y velocidad de cada una de las partículas como resultado de una iteración del sistema.

* El método `ejecutar` no recibe parámetros de entrada y como se ha indicado, lleva a cabo la ejecución del algoritmo el número de iteraciones indicado, y devuelve las coordenadas de maximización de la función así como su valor.


In [5]:
class PSO():
    
    def __init__(self,f,nparticles,niterations,W,c1,c2,a,ndims):
        self.f,self.nparticles,self.niterations,self.W,self.c1,self.c2,self.a,self.ndims=f,nparticles,niterations,W,c1,c2,a,ndims
        self.enjambre = [Partícula(self.W,self.c1,self.c2,self.f,self.a,self.ndims) for _ in range(nparticles)]
        self.bestglobal = -float("infinity")
        self.bestglobalpos= [random.uniform(-(self.a),self.a) for _ in range (self.ndims)]
        
    def actualiza_enjambre(self):
        for p in self.enjambre:
            bestparticula,bestposparticula=p.actualiza(self.bestglobal)
            if bestparticula>self.bestglobal:
                self.bestglobal, self.bestglobalpos= bestparticula,bestposparticula
                
    def ejecutar(self):
        for _ in range(self.niterations):
            self.actualiza_enjambre()
        return self.bestglobalpos,self.bestglobal

### Ejercicio 3 [0.5 ptos]
Evalúa el funcionamiento de las clases anteriores tratando de encontrar las coordenadas de maximización de las siguientes funciones:
    
1. $f(x)=-(x-3)^2+8$
2. $g(x,y)= (x-1)^2 + (y-2)^2+12$
3. $h(x,y,z)= 1 - |x + y + z|$

para ello crea sistemas con 500 partículas, 100 iteraciones, $W=0.5$, $c1=0.3$ y $c2=0.4$ y $a=1$

Llama p1, p2 y p3 a los tres sistemas obtenidos con las funciones anteriores.

In [4]:
def f1(p):
    [x]=p
    return -(x-3)**2+8 

def f2(p):
    [x,y]=p
    return (x-1)**2 + (y-2)**2 +12

def f3(p):
    [x,y,z]=p
    return 1 - abs(x+y+z)

p1 = PSO(f1,500,100,0.5,0.3,0.4,1,1)
p2 = PSO(f2,500,100,0.5,0.3,0.4,1,1)
p3 = PSO(f3,500,100,0.5,0.3,0.4,1,1)

In [5]:
p1.ejecutar()
# El máximo lo alcanza en x=3, con valor 8 (debe dar un resultado aproximado) 

TypeError: 'float' object is not iterable

In [ ]:
p2.ejecutar()
# El máximo lo alcanza en (1,2) con valor 12 (debe dar un resultado aproximado)


In [ ]:
p3.ejecutar()
# El máximo lo alcanza en cualquier terna cuyas coordenadas sumen 0, con valor 1 (debe dar un resultado aproximado)